#### Import all libraries

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import timedelta
from pathlib import Path
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

#### Paths

Now we need to import our dataset. For this, we need to mention the full part of the .csv file that we are going to use.

In [8]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "final_dataset.csv"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

print(OUTPUT_DIR)

C:\Users\amith\Documents\GitHub\resume-projects\Predictive Forecasting of Care Load & Placement Demand\outputs


#### 1. Load data

Importing our dataset, sorting the dataset by date in ascending order

In [9]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df.sort_values(by=["Date"], inplace=True)
df.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-402.0,NaN,NaN,NaN,3,1,0
1,2023-01-13,33.0,53.0,34.0,6622.0,415.0,33.0,53.0,34.0,6566.0,...,NaN,NaN,NaN,-381.0,NaN,NaN,NaN,4,1,0
2,2023-01-14,33.0,52.0,35.0,6677.0,394.0,33.0,53.0,34.0,6622.0,...,NaN,NaN,NaN,-359.0,NaN,NaN,NaN,5,1,1
3,2023-01-15,33.0,52.0,36.0,6733.0,373.0,33.0,52.0,35.0,6677.0,...,NaN,NaN,NaN,-337.0,NaN,NaN,NaN,6,1,1
4,2023-01-16,33.0,51.0,36.0,6788.0,352.0,33.0,52.0,36.0,6733.0,...,NaN,NaN,NaN,-316.0,NaN,NaN,NaN,0,1,0
5,2023-01-17,32.0,51.0,36.0,6844.0,332.0,33.0,51.0,36.0,6788.0,...,NaN,NaN,NaN,-296.0,NaN,NaN,NaN,1,1,0
6,2023-01-18,32.0,51.0,37.0,6900.0,311.0,32.0,51.0,36.0,6844.0,...,NaN,NaN,NaN,-274.0,-337.857143,NaN,46.063104,2,1,0


Dropping all null values and resetting index

In [10]:
df_ml = df.dropna().reset_index(drop=True)
df_ml.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-26,21.0,28.0,34.0,7522.0,211.0,20.0,22.0,41.0,7538.0,...,37.714286,7035.071429,282.714286,-177.0,-173.571429,-245.000000,38.973740,3,1,0
1,2023-01-27,22.0,34.0,26.0,7505.0,242.0,21.0,28.0,34.0,7522.0,...,37.142857,7098.142857,270.357143,-216.0,-171.428571,-233.214286,35.552108,4,1,0
2,2023-01-28,22.0,39.0,18.0,7488.0,272.0,22.0,34.0,26.0,7505.0,...,35.928571,7156.071429,261.642857,-254.0,-177.714286,-225.714286,45.893770,5,1,1
3,2023-01-29,23.0,45.0,11.0,7472.0,303.0,22.0,39.0,18.0,7488.0,...,34.142857,7208.857143,256.642857,-292.0,-192.571429,-222.500000,63.308391,6,1,1
4,2023-01-30,34.0,54.0,29.0,7743.0,196.0,23.0,45.0,11.0,7472.0,...,33.642857,7277.071429,245.500000,-167.0,-196.142857,-211.857143,60.628455,0,1,0
5,2023-01-31,26.0,36.0,36.0,7803.0,158.0,34.0,54.0,29.0,7743.0,...,33.642857,7345.571429,233.071429,-122.0,-195.285714,-199.428571,61.783801,1,1,0
6,2023-02-01,25.0,32.0,27.0,7903.0,231.0,26.0,36.0,36.0,7803.0,...,32.928571,7417.214286,227.357143,-204.0,-204.571429,-194.428571,56.579906,2,2,0


In [11]:
dd = pd.read_csv("C:\\Users\\amith\\Documents\\GitHub\\resume-projects\\Predictive Forecasting of Care Load & Placement Demand\\HHS_Unaccompanied_Alien_Children_Program.csv")
dd["Date"] = pd.to_datetime(dd["Date"])

print(dd["Date"].min())
print(dd["Date"].max())

2023-01-12 00:00:00
2025-12-21 00:00:00


In [12]:
print("Data loaded.")
print()
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Total days:", len(df))
print()
print("Date range after lag NaNs removed:", df_ml["Date"].min().date(), "to", df_ml["Date"].max().date())
print("Days usable for feature models (after lag NaNs removed):", len(df_ml))

Data loaded.

Date range: 2023-01-12 to 2025-12-21
Total days: 1075

Date range after lag NaNs removed: 2023-01-26 to 2025-12-21
Days usable for feature models (after lag NaNs removed): 1061


#### 2. Time-based train / validation / test split
    (~67% train, ~25% val, rest test) — same idea as the notebook

In [13]:
initial_day = df["Date"].iloc[0]
final_day = df["Date"].iloc[-1]
total_days = (final_day - initial_day).days

print(total_days)

1074


In [14]:
n_train = round(total_days * 0.67)
n_val = round(total_days * 0.25)


train_end = initial_day + timedelta(days = n_train - 1)
val_end = train_end + timedelta(days = n_val - 1)

print("Training dates range from : ",initial_day,"to",train_end)
print("Validation dates ranges from : ",train_end+timedelta(1),"to",val_end)

Training dates range from :  2023-01-12 00:00:00 to 2024-12-31 00:00:00
Validation dates ranges from :  2025-01-01 00:00:00 to 2025-09-24 00:00:00


In [15]:
train_df = df[df["Date"] <= train_end].copy()
val_df = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
test_df = df[df["Date"] > val_end].copy()

train_ml = df_ml[df_ml["Date"] <= train_end].copy()
val_ml = df_ml[(df_ml["Date"] > train_end) & (df_ml["Date"] <= val_end)].copy()
test_ml = df_ml[df_ml["Date"] > val_end].copy()

In [16]:
print("\nSplit sizes:")
print("  Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))


Split sizes:
  Train: 720 | Val: 267 | Test: 88


#### Helper: evaluation metrics

In [17]:
def calc_metrics(actual, predicted):
    """Return MAE, RMSE, MAPE for one series of forecasts."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    nonzero = actual != 0
    if nonzero.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100
    return round(float(mae), 2), round(float(rmse), 2), round(float(mape), 2)

In [18]:
def accuracy_pct(mape):
    """Simple forecast accuracy % from MAPE."""
    if pd.isna(mape):
        return np.nan
    return round(max(0, 100 - mape), 2)

#### 3. Baseline models

In [19]:
def naive_forecast(series, horizon):
    """Repeat the last observed value for every future day."""
    last_value = float(series.iloc[-1])
    return np.full(horizon, last_value)

In [38]:
naive_forecast(train_df["HHS_Care"],7)

array([6430., 6430., 6430., 6430., 6430., 6430., 6430.])

In [39]:
def moving_average_forecast(series, horizon, window=7):
    """Use the recent rolling mean as the constant forecast."""
    avg = float(series.tail(window).mean())
    return np.full(horizon, avg)

In [40]:
moving_average_forecast(train_df["HHS_Care"],7)

array([6559.57142857, 6559.57142857, 6559.57142857, 6559.57142857,
       6559.57142857, 6559.57142857, 6559.57142857])

#### 4. ARIMA

In [41]:
def arima_forecast(series, horizon, order=(2, 1, 2)):
    """Fit ARIMA and forecast horizon steps ahead."""
    try:
        model = ARIMA(series.astype(float), order=order)
        fitted = model.fit()
        pred = fitted.forecast(steps=horizon)
        return np.asarray(pred, dtype=float)
    except Exception as e:
        print("  ARIMA failed:", e)
        return naive_forecast(series, horizon)

In [42]:
arima_forecast(train_df["HHS_Care"],10)

array([6471.64215182, 6498.18257942, 6491.98930537, 6460.56066979,
       6428.80028372, 6419.52555851, 6437.29208399, 6466.82957236,
       6485.84782431, 6481.68396183])

#### 5. Exponential Smoothing (Holt linear trend)

In [43]:
def exp_smoothing_forecast(series, horizon):
    """Holt linear trend (no strong weekly seasonality assumed)."""
    try:
        model = ExponentialSmoothing(
            series.astype(float),
            trend="add",
            seasonal=None,
            initialization_method="estimated"
        )
        fitted = model.fit(optimized=True)
        pred = fitted.forecast(horizon)
        return np.asarray(pred, dtype=float)
    except Exception as e:
        print("  ExpSmoothing failed:", e)
        return naive_forecast(series, horizon)

In [44]:
exp_smoothing_forecast(train_df["HHS_Care"], 7)

array([6405.88017775, 6381.76035549, 6357.64053324, 6333.52071099,
       6309.40088873, 6285.28106648, 6261.16124423])

#### 6. Machine learning models (Random Forest & Gradient Boosting)

In [45]:
FEATURE_COLS = [
    "Placed_In_CBP_lag_1", "CBP_Custody_lag_1", "CBP_To_HHS_lag_1",
    "HHS_Care_lag_1", "Discharged_From_HHS_lag_1",
    "Placed_In_CBP_lag_7", "CBP_Custody_lag_7", "CBP_To_HHS_lag_7",
    "HHS_Care_lag_7", "Discharged_From_HHS_lag_7",
    "Placed_In_CBP_lag_14", "CBP_Custody_lag_14", "CBP_To_HHS_lag_14",
    "HHS_Care_lag_14", "Discharged_From_HHS_lag_14",
    "Placed_In_CBP_rolling_avg_7", "CBP_Custody_rolling_avg_7",
    "CBP_To_HHS_rolling_avg_7", "HHS_Care_rolling_avg_7",
    "Discharged_From_HHS_rolling_avg_7",
    "Placed_In_CBP_rolling_avg_14", "CBP_Custody_rolling_avg_14",
    "CBP_To_HHS_rolling_avg_14", "HHS_Care_rolling_avg_14",
    "Discharged_From_HHS_rolling_avg_14",
    "Net_HHS_Flow", "Net_HHS_Flow_rolling_avg_7", "Net_HHS_Flow_rolling_avg_14",
    "Net_HHS_Flow_rolling_std_7",
    "Day_Of_Week", "Month", "Is_Weekend"
]

In [46]:
def train_ml_model(train_frame, target, model_type="rf"):
    """Train Random Forest or Gradient Boosting on feature columns."""
    X = train_frame[FEATURE_COLS]
    y = train_frame[target]
    if model_type == "rf":
        model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    else:
        model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model

In [47]:
def ml_predict(model, frame):
    """Predict using an already trained ML model."""
    return model.predict(frame[FEATURE_COLS])

In [48]:
val_ml

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
706,2025-01-01,25.0,66.0,94.0,6405.0,144.0,62.0,111.0,116.0,6430.0,...,170.000000,6563.000000,192.000000,-50.0,-37.714286,-22.000000,31.244809,2,1,0
707,2025-01-02,35.0,70.0,53.0,6292.0,170.0,25.0,66.0,94.0,6405.0,...,159.214286,6541.000000,187.142857,-117.0,-56.857143,-27.928571,33.128178,3,1,0
708,2025-01-03,38.0,71.0,57.0,6155.0,175.0,35.0,70.0,53.0,6292.0,...,147.928571,6509.357143,181.928571,-118.0,-71.428571,-34.000000,34.563262,4,1,0
709,2025-01-04,40.0,72.0,60.0,6019.0,180.0,38.0,71.0,57.0,6155.0,...,136.142857,6468.071429,176.357143,-120.0,-81.428571,-40.214286,37.344599,5,1,1
710,2025-01-05,43.0,73.0,64.0,5882.0,185.0,40.0,72.0,60.0,6019.0,...,123.857143,6417.142857,170.428571,-121.0,-86.857143,-46.571429,40.259279,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,2025-09-20,12.0,41.0,12.0,2121.0,8.0,16.0,46.0,12.0,2109.0,...,13.571429,2068.714286,9.642857,4.0,1.000000,3.928571,5.354126,5,9,1
969,2025-09-21,8.0,35.0,13.0,2134.0,5.0,12.0,41.0,12.0,2121.0,...,13.857143,2076.428571,9.500000,8.0,1.000000,4.357143,5.354126,6,9,1
970,2025-09-22,17.0,40.0,14.0,2141.0,10.0,8.0,35.0,13.0,2134.0,...,14.500000,2084.714286,9.785714,4.0,1.571429,4.714286,5.442338,0,9,0
971,2025-09-23,11.0,36.0,10.0,2145.0,15.0,17.0,40.0,14.0,2141.0,...,14.428571,2093.428571,9.857143,-5.0,0.142857,4.571429,5.698789,1,9,0


In [49]:
model = train_ml_model(train_ml, "HHS_Care", model_type="rf")
pred = ml_predict(model, val_ml.iloc[0:7])
pred

array([6377.1 , 6380.67, 6241.07, 6077.28, 5984.54, 5847.86, 5802.22])

In [50]:
model = train_ml_model(train_ml, "HHS_Care", model_type="gb")
pred = ml_predict(model, val_ml.iloc[0:7])
pred

array([6419.6686367 , 6359.20908644, 6172.85428283, 6062.56989778,
       5931.29993869, 5869.41915424, 5811.74771306])

#### 7. Evaluate models on the TEST set

In [53]:
TARGETS = ["HHS_Care", "Discharged_From_HHS"]
HORIZON = len(test_df)

print(TARGETS)

['HHS_Care', 'Discharged_From_HHS']


In [54]:
results = []
forecast_store = {}

for target in TARGETS:
    print("\n" + "=" * 60)
    print("Target:", target)
    print("=" * 60)

    train_series = train_df.set_index("Date")[target]
    actual_test = test_df[target].values.astype(float)

    # Naive
    pred = naive_forecast(train_series, HORIZON)
    mae, rmse, mape = calc_metrics(actual_test, pred)
    results.append({"Target": target, "Model": "Naive", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "Naive")] = pred
    print(f"  Naive              MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")

    # Moving Average
    pred = moving_average_forecast(train_series, HORIZON, window=7)
    mae, rmse, mape = calc_metrics(actual_test, pred)
    results.append({"Target": target, "Model": "Moving_Average", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "Moving_Average")] = pred
    print(f"  Moving Average     MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")

    # ARIMA
    pred = arima_forecast(train_series, HORIZON, order=(2, 1, 2))
    mae, rmse, mape = calc_metrics(actual_test, pred)
    results.append({"Target": target, "Model": "ARIMA", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "ARIMA")] = pred
    print(f"  ARIMA              MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")

    # Exponential Smoothing
    pred = exp_smoothing_forecast(train_series, HORIZON)
    mae, rmse, mape = calc_metrics(actual_test, pred)
    results.append({"Target": target, "Model": "ExpSmoothing", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "ExpSmoothing")] = pred
    print(f"  ExpSmoothing       MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")

    # Random Forest
    rf_model = train_ml_model(train_ml, target, model_type="rf")
    pred_rf = ml_predict(rf_model, test_ml)
    actual_ml = test_ml[target].values.astype(float)
    mae, rmse, mape = calc_metrics(actual_ml, pred_rf)
    results.append({"Target": target, "Model": "RandomForest", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "RandomForest")] = pred_rf
    print(f"  RandomForest       MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")

    # Gradient Boosting
    gb_model = train_ml_model(train_ml, target, model_type="gb")
    pred_gb = ml_predict(gb_model, test_ml)
    mae, rmse, mape = calc_metrics(actual_ml, pred_gb)
    results.append({"Target": target, "Model": "GradientBoosting", "MAE": mae, "RMSE": rmse,
                    "MAPE": mape, "Forecast_Accuracy_Pct": accuracy_pct(mape)})
    forecast_store[(target, "GradientBoosting")] = pred_gb
    print(f"  GradientBoosting   MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:6.2f}%")


Target: HHS_Care
  Naive              MAE= 4120.44  RMSE= 4121.68  MAPE=178.94%
  Moving Average     MAE= 4250.01  RMSE= 4251.21  MAPE=184.56%
  ARIMA              MAE= 4147.08  RMSE= 4148.35  MAPE=180.10%
  ExpSmoothing       MAE= 3047.11  RMSE= 3129.37  MAPE=133.53%
  RandomForest       MAE= 3496.28  RMSE= 3497.76  MAPE=151.86%
  GradientBoosting   MAE= 3524.87  RMSE= 3526.51  MAPE=153.10%

Target: Discharged_From_HHS
  Naive              MAE=  149.98  RMSE=  150.03  MAPE=1916.06%
  Moving Average     MAE=  153.69  RMSE=  153.74  MAPE=1962.86%
  ARIMA              MAE=  194.30  RMSE=  195.94  MAPE=2520.10%
  ExpSmoothing       MAE=  131.71  RMSE=  132.10  MAPE=1667.27%
  RandomForest       MAE=  151.26  RMSE=  151.41  MAPE=1901.86%
  GradientBoosting   MAE=  181.22  RMSE=  181.62  MAPE=2283.23%


#### 8. Save metric table

In [56]:
metrics_df = pd.DataFrame(results)
metrics_df

,Target,Model,MAE,RMSE,MAPE,Forecast_Accuracy_Pct
0,HHS_Care,Naive,4120.44,4121.68,178.94,0
1,HHS_Care,Moving_Average,4250.01,4251.21,184.56,0
2,HHS_Care,ARIMA,4147.08,4148.35,180.10,0
3,HHS_Care,ExpSmoothing,3047.11,3129.37,133.53,0
4,HHS_Care,RandomForest,3496.28,3497.76,151.86,0
5,HHS_Care,GradientBoosting,3524.87,3526.51,153.10,0
6,Discharged_From_HHS,Naive,149.98,150.03,1916.06,0
7,Discharged_From_HHS,Moving_Average,153.69,153.74,1962.86,0
8,Discharged_From_HHS,ARIMA,194.30,195.94,2520.10,0
9,Discharged_From_HHS,ExpSmoothing,131.71,132.10,1667.27,0


#### 9. 14-day forward forecast from the last observed day

In [60]:
FORWARD_DAYS = 14
last_date = df["Date"].max()
future_dates = pd.date_range(last_date + timedelta(days=1), periods=FORWARD_DAYS, freq="D")

forward_rows = []

for target in TARGETS:
    full_series = df.set_index("Date")[target]

    fwd_naive = naive_forecast(full_series, FORWARD_DAYS)
    fwd_ma = moving_average_forecast(full_series, FORWARD_DAYS, window=7)
    fwd_arima = arima_forecast(full_series, FORWARD_DAYS, order=(2, 1, 2))
    fwd_es = exp_smoothing_forecast(full_series, FORWARD_DAYS)

    # ML: retrain on all history; hold last feature row constant for short horizon
    rf_full = train_ml_model(df_ml, target, model_type="rf")
    gb_full = train_ml_model(df_ml, target, model_type="gb")
    last_feat = df_ml[FEATURE_COLS].iloc[[-1]]
    future_feat = pd.concat([last_feat] * FORWARD_DAYS, ignore_index=True)
    fwd_rf = rf_full.predict(future_feat)
    fwd_gb = gb_full.predict(future_feat)

    for i, d in enumerate(future_dates):
        forward_rows.append({
            "Date": d,
            "Target": target,
            "Naive": round(float(fwd_naive[i]), 1),
            "Moving_Average": round(float(fwd_ma[i]), 1),
            "ARIMA": round(float(fwd_arima[i]), 1),
            "ExpSmoothing": round(float(fwd_es[i]), 1),
            "RandomForest": round(float(fwd_rf[i]), 1),
            "GradientBoosting": round(float(fwd_gb[i]), 1),
        })

forward_df = pd.DataFrame(forward_rows)
forward_df

,Date,Target,Naive,Moving_Average,ARIMA,ExpSmoothing,RandomForest,GradientBoosting
0,2025-12-22,HHS_Care,2484.0,2475.9,2486.4,2488.0,2481.0,2475.1
1,2025-12-23,HHS_Care,2484.0,2475.9,2486.4,2491.9,2481.0,2475.1
2,2025-12-24,HHS_Care,2484.0,2475.9,2484.6,2495.9,2481.0,2475.1
3,2025-12-25,HHS_Care,2484.0,2475.9,2482.4,2499.9,2481.0,2475.1
4,2025-12-26,HHS_Care,2484.0,2475.9,2481.4,2503.8,2481.0,2475.1
5,2025-12-27,HHS_Care,2484.0,2475.9,2482.1,2507.8,2481.0,2475.1
6,2025-12-28,HHS_Care,2484.0,2475.9,2483.8,2511.7,2481.0,2475.1
7,2025-12-29,HHS_Care,2484.0,2475.9,2485.0,2515.7,2481.0,2475.1
8,2025-12-30,HHS_Care,2484.0,2475.9,2485.1,2519.7,2481.0,2475.1
9,2025-12-31,HHS_Care,2484.0,2475.9,2484.2,2523.6,2481.0,2475.1


#### 10. Net pressure early warning (Transfers - Discharges)

In [59]:
transfer_series = df.set_index("Date")["CBP_To_HHS"]
discharge_series = df.set_index("Date")["Discharged_From_HHS"]

fwd_transfers = arima_forecast(transfer_series, FORWARD_DAYS, order=(2, 1, 2))
fwd_discharges = arima_forecast(discharge_series, FORWARD_DAYS, order=(2, 1, 2))
net_pressure = fwd_transfers - fwd_discharges

pressure_df = pd.DataFrame({
    "Date": future_dates,
    "Forecast_Transfers_In": np.round(fwd_transfers, 1),
    "Forecast_Discharges": np.round(fwd_discharges, 1),
    "Net_Pressure": np.round(net_pressure, 1),
    "Capacity_Stress_Flag": np.where(
        net_pressure > 0, "Rising care load risk", "Discharge keeping pace"
    )
})

pressure_df

,Date,Forecast_Transfers_In,Forecast_Discharges,Net_Pressure,Capacity_Stress_Flag
0,2025-12-22,11.1,12.7,-1.6,Discharge keeping pace
1,2025-12-23,9.7,13.9,-4.3,Discharge keeping pace
2,2025-12-24,10.5,16.6,-6.1,Discharge keeping pace
3,2025-12-25,9.4,18.8,-9.4,Discharge keeping pace
4,2025-12-26,10.4,18.9,-8.5,Discharge keeping pace
5,2025-12-27,9.4,16.8,-7.5,Discharge keeping pace
6,2025-12-28,10.4,14.3,-3.9,Discharge keeping pace
7,2025-12-29,9.4,13.1,-3.8,Discharge keeping pace
8,2025-12-30,10.3,14.1,-3.8,Discharge keeping pace
9,2025-12-31,9.4,16.5,-7.1,Discharge keeping pace


#### 11. KPI summary (best model per target by MAE)

In [61]:
kpi_rows = []
for target in TARGETS:
    sub = metrics_df[metrics_df["Target"] == target]
    best = sub.loc[sub["MAE"].idxmin()]
    kpi_rows.append({
        "Target": target,
        "Best_Model": best["Model"],
        "Best_MAE": best["MAE"],
        "Best_RMSE": best["RMSE"],
        "Best_MAPE": best["MAPE"],
        "Forecast_Accuracy_Pct": best["Forecast_Accuracy_Pct"],
        "Surge_Lead_Time_Days": FORWARD_DAYS,
        "Notes": "Lower MAE/RMSE is better. Accuracy % = 100 - MAPE."
    })

kpi_df = pd.DataFrame(kpi_rows)
kpi_df

,Target,Best_Model,Best_MAE,Best_RMSE,Best_MAPE,Forecast_Accuracy_Pct,Surge_Lead_Time_Days,Notes
0,HHS_Care,ExpSmoothing,3047.11,3129.37,133.53,0,14,Lower MAE/RMSE is better. Accuracy % = 100 - M...
1,Discharged_From_HHS,ExpSmoothing,131.71,132.10,1667.27,0,14,Lower MAE/RMSE is better. Accuracy % = 100 - M...


#### 12. Test-period predictions for charts

In [62]:
test_pred_rows = []
model_names = ["Naive", "Moving_Average", "ARIMA", "ExpSmoothing", "RandomForest", "GradientBoosting"]

for target in TARGETS:
    for model_name in model_names:
        preds = forecast_store.get((target, model_name))
        if preds is None:
            continue
        if model_name in ("RandomForest", "GradientBoosting"):
            dates = test_ml["Date"].values
            actual = test_ml[target].values
        else:
            dates = test_df["Date"].values
            actual = test_df[target].values
        n = min(len(dates), len(preds), len(actual))
        for i in range(n):
            test_pred_rows.append({
                "Date": dates[i],
                "Target": target,
                "Model": model_name,
                "Actual": round(float(actual[i]), 1),
                "Predicted": round(float(preds[i]), 1)
            })

test_pred_df = pd.DataFrame(test_pred_rows)
test_pred_df

,Date,Target,Model,Actual,Predicted
0,2025-09-25,HHS_Care,Naive,2132.0,6430.0
1,2025-09-26,HHS_Care,Naive,2137.0,6430.0
2,2025-09-27,HHS_Care,Naive,2141.0,6430.0
3,2025-09-28,HHS_Care,Naive,2146.0,6430.0
4,2025-09-29,HHS_Care,Naive,2148.0,6430.0
...,...,...,...,...,...
1051,2025-12-17,Discharged_From_HHS,GradientBoosting,10.0,177.4
1052,2025-12-18,Discharged_From_HHS,GradientBoosting,16.0,210.1
1053,2025-12-19,Discharged_From_HHS,GradientBoosting,15.0,208.9
1054,2025-12-20,Discharged_From_HHS,GradientBoosting,15.0,208.9
